# Tutorial 8: Technical Indicator Plugins

Spectral-Env-Core has a plugin system for technical indicators. Instead of hardcoding features into the environment, you pass indicator objects at construction time and the observation space expands automatically.

This tutorial covers:
1. Using built-in indicators
2. How indicators change the observation space
3. Writing a custom indicator from scratch
4. Training an agent with indicator-enriched observations

---

## Why Indicators as Plugins?

Different trading strategies rely on different signals. A momentum trader wants RSI and MACD. A mean-reversion trader wants Bollinger Bands. A volatility trader wants ATR.

Hardcoding all of these into the observation space wastes capacity — the agent has to learn to ignore signals it doesn't need. The plugin system lets you compose exactly the observation space your strategy requires:

```python
# Momentum strategy
env = SpectralTradingEnv(indicators=[RSI(14), MACD(), SMA_Crossover(10, 30)])

# Volatility strategy
env = SpectralTradingEnv(indicators=[ATR(14), BollingerBands(20)])

# Kitchen sink (let the network figure it out)
env = SpectralTradingEnv(indicators=[RSI(14), MACD(), BollingerBands(20), ATR(14), SMA_Crossover(10, 30)])
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spectral_env_core import SpectralTradingEnv
from spectral_env_core.indicators import RSI, MACD, BollingerBands, ATR, SMA_Crossover

## 1. Built-in Indicators in Action

In [ ]:
# Environment WITHOUT indicators
env_plain = SpectralTradingEnv(num_assets=2, num_steps=252, initial_price=[150.0, 300.0])

# Environment WITH indicators
env_indicators = SpectralTradingEnv(
    num_assets=2,
    num_steps=252,
    initial_price=[150.0, 300.0],
    indicators=[
        RSI(period=14),
        MACD(fast=12, slow=26, signal=9),
        BollingerBands(period=20, std_dev=2),
        ATR(period=14),
    ],
)

print(f"Plain observation space:      {env_plain.observation_space.shape[0]} dims")
print(f"With indicators:             {env_indicators.observation_space.shape[0]} dims")
print(f"")
print(f"Breakdown:")
print(f"  Price windows:  {env_indicators.num_price_features} (2 assets × 30 steps)")
print(f"  Meta features:  {env_indicators.num_meta_features} (cash + 2 shares + portfolio + exit_cost + time)")
print(f"  Indicators:     {env_indicators.num_indicator_features}")
print(f"    RSI:            2 (1 per asset)")
print(f"    MACD:           4 (2 per asset: line + signal)")
print(f"    BollingerBands: 2 (1 per asset)")
print(f"    ATR:            2 (1 per asset)")
print(f"  Total:          {env_indicators.observation_space.shape[0]}")

In [ ]:
# Visualise indicator values over an episode
env_vis = SpectralTradingEnv(
    num_assets=1, num_steps=252, initial_price=150.0,
    volatility=0.25, drift=0.10,
    garch_alpha=0.08, garch_beta=0.85,
    indicators=[RSI(14), BollingerBands(20), ATR(14)],
)
env_vis.reset(seed=42)

# Collect indicator values at each step
prices_hist = []
rsi_hist = []
bb_hist = []
atr_hist = []

for step in range(253):  # 0 to 252
    prices_hist.append(env_vis.brownian_path[step, 0])
    rsi_val = RSI(14).compute(env_vis.brownian_path, step)
    bb_val  = BollingerBands(20).compute(env_vis.brownian_path, step)
    atr_val = ATR(14).compute(env_vis.brownian_path, step)
    rsi_hist.append(rsi_val[0])
    bb_hist.append(bb_val[0])
    atr_hist.append(atr_val[0])

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(prices_hist, color='black', lw=1.5)
axes[0].set_title('Price', fontsize=11)
axes[0].set_ylabel('$')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rsi_hist, color='purple', lw=1.2)
axes[1].axhline(0.6, color='red', ls='--', lw=0.8, alpha=0.7, label='Overbought')
axes[1].axhline(-0.6, color='green', ls='--', lw=0.8, alpha=0.7, label='Oversold')
axes[1].axhline(0, color='gray', ls='-', lw=0.5)
axes[1].set_title('RSI(14) — normalised [-1, +1]', fontsize=11)
axes[1].set_ylim(-1.1, 1.1)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

axes[2].plot(bb_hist, color='steelblue', lw=1.2)
axes[2].axhline(1, color='red', ls='--', lw=0.8, alpha=0.7, label='Upper band')
axes[2].axhline(-1, color='green', ls='--', lw=0.8, alpha=0.7, label='Lower band')
axes[2].axhline(0, color='gray', ls='-', lw=0.5)
axes[2].set_title('Bollinger Band Position — normalised', fontsize=11)
axes[2].set_ylim(-2.2, 2.2)
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

axes[3].plot(atr_hist, color='darkorange', lw=1.2)
axes[3].set_title('ATR(14) — normalised volatility', fontsize=11)
axes[3].set_xlabel('Step')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("These values are computed at every step and appended to the observation.")
print("The agent sees them alongside the price window and portfolio state.")

## 2. Writing a Custom Indicator

Any indicator can be added by subclassing `Indicator`. You need three things:

1. `name` — unique string identifier
2. `n_features` — how many values it produces per asset
3. `compute(prices, step)` — the actual computation

Here's a complete example: **Rate of Change (ROC)** — the percentage price change over N periods.

In [ ]:
from spectral_env_core.indicators import Indicator

class ROC(Indicator):
    """
    Rate of Change — percentage price change over `period` steps.
    
    Normalised by dividing by 10 so typical values are in [-1, 1] range
    for a stock with ~20-30% annual volatility.
    """
    
    def __init__(self, period: int = 10):
        self.period = period
    
    @property
    def name(self) -> str:
        return f"roc_{self.period}"
    
    @property
    def n_features(self) -> int:
        return 1  # one value per asset
    
    def compute(self, prices: np.ndarray, step: int) -> np.ndarray:
        num_assets = prices.shape[1]
        result = np.zeros(num_assets, dtype=np.float32)
        
        for i in range(num_assets):
            if step < self.period:
                continue  # not enough history
            
            current = prices[step, i]
            past    = prices[step - self.period, i]
            
            if past > 0:
                roc_pct = (current - past) / past * 100
                # Normalise: divide by 10 so ~20% vol gives values in [-2, 2]
                result[i] = np.clip(roc_pct / 10.0, -3.0, 3.0)
        
        return result


# Test it
env_custom = SpectralTradingEnv(
    num_assets=1, num_steps=252, initial_price=100.0,
    indicators=[ROC(period=5), ROC(period=20)],  # two ROC at different periods
)

print(f"Observation space with custom ROC indicators: {env_custom.observation_space.shape[0]} dims")
print(f"  Added: ROC(5) = 1 feature + ROC(20) = 1 feature = +2 dims")

# Run a step and see the values
obs, info = env_custom.reset(seed=0)
for _ in range(30):  # advance past the lookback
    obs, _, _, _, info = env_custom.step(np.array([0.0]))

print(f"\nLast 4 obs values (exit_cost, time_remaining, ROC_5, ROC_20):")
print(f"  {obs[-4:]}")  

### Custom Indicator Template

```python
from spectral_env_core.indicators import Indicator
import numpy as np

class MyIndicator(Indicator):
    def __init__(self, param1: int = 10):
        self.param1 = param1
    
    @property
    def name(self) -> str:
        return f"my_indicator_{self.param1}"
    
    @property
    def n_features(self) -> int:
        return 1  # values per asset
    
    @property
    def per_asset(self) -> bool:
        return True  # set False for global indicators (e.g. cross-asset spread)
    
    def compute(self, prices: np.ndarray, step: int) -> np.ndarray:
        # prices shape: (num_steps+1, num_assets)
        num_assets = prices.shape[1]
        result = np.zeros(num_assets, dtype=np.float32)
        
        for i in range(num_assets):
            # Your computation here
            # Use prices[:step+1, i] for history up to current step
            result[i] = ...  # normalised to roughly [-1, 1]
        
        return result
```

**Key rules:**
- Always return `float32` array
- Normalise output to approximately [-1, 1] or [-2, 2]
- Handle edge cases (not enough history → return 0.0)
- Use only `prices[:step+1]` — don't look ahead!

## 3. Training with Indicators

Indicators plug seamlessly into the training pipeline. The `SpectralExtractor` auto-detects the expanded observation space — its `meta_net` input dimension adjusts automatically.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from spectral_env_core import SpectralExtractor, EntropyCoefficientCallback

# 3-asset environment with momentum indicators
env_kwargs = dict(
    num_assets=3,
    num_steps=252,
    initial_price=[180.0, 300.0, 95.0],
    drift=[0.15, 0.22, 0.08],
    volatility=[0.28, 0.32, 0.18],
    garch_alpha=0.08,
    garch_beta=0.85,
    jump_intensity=1.5,
    jump_mean=-0.03,
    jump_std=0.05,
    starting_cash=100_000,
    max_shares=500,
    max_trade_size=50,
    correlation=np.array([
        [1.0, 0.5, 0.2],
        [0.5, 1.0, 0.3],
        [0.2, 0.3, 1.0],
    ]),
    indicators=[
        RSI(period=14),
        MACD(fast=12, slow=26, signal=9),
        BollingerBands(period=20),
        ATR(period=14),
        SMA_Crossover(fast=10, slow=30),
    ],
)

# Check dims
test_env = SpectralTradingEnv(**env_kwargs)
obs_dim = test_env.observation_space.shape[0]
print(f"Observation space: {obs_dim} dims")
print(f"  Price windows:  {test_env.num_price_features}")
print(f"  Meta features:  {test_env.num_meta_features}")
print(f"  Indicators:     {test_env.num_indicator_features}")
print(f"")
print(f"SpectralExtractor will auto-size meta_net input to {obs_dim - test_env.num_price_features} dims")

In [ ]:
# Set up parallel training
train_vec = make_vec_env(SpectralTradingEnv, n_envs=4, env_kwargs=env_kwargs, vec_env_cls=SubprocVecEnv)
train_vec = VecNormalize(train_vec, norm_obs=True, norm_reward=False, clip_obs=10.0)

# SpectralExtractor auto-adapts to the indicator-expanded obs space
policy_kwargs = dict(
    log_std_init=-1.0,
    features_extractor_class=SpectralExtractor,
    features_extractor_kwargs=dict(
        num_assets=3,
        lookback_window=30,
        asset_embed_dim=32,
        meta_embed_dim=32,  # larger projection since we have more meta+indicator features
    ),
    net_arch=dict(pi=[64, 64], vf=[128, 128, 64]),
)

ent_cb = EntropyCoefficientCallback(start=0.005, end=0.001)

model = PPO(
    "MlpPolicy", train_vec, verbose=1, device="cpu",
    n_steps=2048, batch_size=128, n_epochs=10,
    learning_rate=3e-5, gamma=0.995, gae_lambda=0.90,
    clip_range=0.2, target_kl=0.05, ent_coef=0.005, vf_coef=0.75,
    policy_kwargs=policy_kwargs,
)

print(f"\nTotal parameters: {sum(p.numel() for p in model.policy.parameters()):,}")
print(f"Training for 200K steps (demo)...")
model.learn(total_timesteps=200_000, callback=[ent_cb])
train_vec.close()
print("Done.")

In [ ]:
# Quick eval
eval_env = SpectralTradingEnv(**env_kwargs)
rewards = []
for ep in range(15):
    obs, info = eval_env.reset(seed=500 + ep)
    total = 0.0
    for _ in range(252):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total += reward
        if terminated or truncated:
            break
    rewards.append(total)

print(f"\nEval (15 episodes):")
print(f"  Mean reward: {np.mean(rewards):+.2f}%")
print(f"  Std:         {np.std(rewards):.2f}%")

## Design Notes

### Why plugins instead of hardcoded features?

1. **Observation space bloat** — a 5-asset env with all indicators hardcoded would have 200+ obs dims even if the agent only uses RSI. Plugins let you include exactly what you need.

2. **Research flexibility** — testing whether MACD helps vs hurts is a one-line change, not a code fork.

3. **Custom strategies** — institutional quants have proprietary indicators. The plugin system lets them integrate without modifying core code.

### How it works internally

```
Observation layout:
[price_window_0, ..., price_window_N, meta_features, indicator_0, indicator_1, ...]
 ├── num_price_features ──┤  ├── meta ──┤  ├──── num_indicator_features ────┤
```

- At `__init__`, the env counts total indicator features and sizes `observation_space`
- At `_get_obs()`, each indicator's `compute()` is called and appended to the obs vector
- `SpectralExtractor` treats everything after the price windows as `meta_net` input — indicators are projected alongside portfolio state through the same network

### Performance

Indicators are computed from the pre-generated `brownian_path` array — they're pure NumPy lookups on data that already exists in memory. Typical overhead is <5% of step time even with 5 indicators on 5 assets.

---

### Available Built-in Indicators

| Indicator | Features/asset | Signal |
|---|---|---|
| `RSI(period=14)` | 1 | Overbought/oversold [-1, +1] |
| `MACD(fast=12, slow=26, signal=9)` | 2 | Trend momentum + crossover |
| `BollingerBands(period=20, std_dev=2)` | 1 | Position within volatility bands |
| `ATR(period=14)` | 1 | Current volatility level |
| `SMA_Crossover(fast=10, slow=30)` | 1 | Moving average trend direction |